# Ensemble Monte Carlo Uncertainty Quantification

This notebook demonstrates the complete PyPath ensemble workflow for Ecopath mass-balance models.

**Goal:** Understand how parameter uncertainty propagates through the food web model by running 50 replicate scenarios with sampled parameters.

**Duration:** ~5 minutes execution time

## Cell 2: Imports

Import PyPath ensemble functions and data manipulation libraries.

In [ ]:
# Core ensemble and modeling functions
from pypath.core.ensemble import (
    run_ensemble_rpath,
    get_ensemble_confidence_intervals,
    ensemble_statistics,
    EnsembleResult
)
from pypath.core.params import create_rpath_params, RpathParams
from pypath.core.sampling import (
    SampleContainer,
    EcopathGroupSample,
    validate_samples
)

# Data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("✓ All imports successful")

## Cell 3: Load Example Model

Load the 12-group coastal example model from CSV files.

In [ ]:
# Locate example model data
import pypath
pypath_dir = Path(pypath.__file__).parent
example_dir = pypath_dir.parent / "example_model_data"

# Load model
params = create_rpath_params(
    model_csv=str(example_dir / "model.csv"),
    diet_csv=str(example_dir / "diet.csv")
)

print(f"Loaded model with {len(params.model)} groups:")
print("\nGroup Information:")
display(params.model[["Group", "Type", "Biomass", "PB", "QB", "EE"]].head(10))

# Summary
print(f"\nTotal system biomass (base model): {params.model['Biomass'].sum():.2f} t/km²")

## Cell 4: Define Sampling Parameters

Create bootstrap samples for 3 key groups with 10% uncertainty (CV = 0.1).

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Configuration
n_replicates = 50
sampling_cv = 0.1  # 10% coefficient of variation
groups_to_sample = ["Zooplankton", "Benthic invertebrates", "Demersal fish"]

# Create container
container = SampleContainer(
    n_replicates=n_replicates,
    random_seed=42
)

# Add samples for each group
for group_name in groups_to_sample:
    # Get base parameters
    group_row = params.model[params.model["Group"] == group_name].iloc[0]
    
    biomass_base = group_row["Biomass"]
    pb_base = group_row["PB"]
    qb_base = group_row["QB"]
    ee_base = group_row["EE"]
    
    # Generate lognormal samples (appropriate for positive-valued parameters)
    biomass_samples = np.random.lognormal(
        mean=np.log(biomass_base),
        sigma=sampling_cv,
        size=n_replicates
    )
    pb_samples = np.random.lognormal(
        mean=np.log(pb_base),
        sigma=sampling_cv,
        size=n_replicates
    )
    qb_samples = np.random.lognormal(
        mean=np.log(qb_base),
        sigma=sampling_cv,
        size=n_replicates
    )
    
    # Beta samples for EE (bounded to [0,1])
    ee_samples = np.random.beta(
        a=ee_base * (1 - ee_base) / (sampling_cv**2) - 1,
        b=(1 - ee_base) * (1 - ee_base) / (sampling_cv**2) - 1,
        size=n_replicates
    ).clip(0, 1)
    
    # Create and add sample
    sample = EcopathGroupSample(
        group_name=group_name,
        biomass_samples=biomass_samples,
        pb_samples=pb_samples,
        qb_samples=qb_samples,
        ee_samples=ee_samples,
        biomass_se=sampling_cv,
        pb_se=sampling_cv,
    )
    container.add_group_sample(sample)

print(f"Created samples for {len(container.group_samples)} groups:")
for group_name, sample in container.group_samples.items():
    print(f"  - {group_name}: {sample.n_replicates} replicates")

## Cell 5: Validate Samples

Inspect sample statistics before running the ensemble.

In [ ]:
# Display sample statistics
print("Sample Statistics (Zooplankton):")
print("="*50)

zoo_sample = container.group_samples["Zooplankton"]

print(f"\nBiomass:")
print(f"  Base: {params.model[params.model['Group']=='Zooplankton']['Biomass'].values[0]:.3f} t/km²")
print(f"  Mean (samples): {zoo_sample.biomass_samples.mean():.3f} ± {zoo_sample.biomass_samples.std():.3f}")
print(f"  Range: {zoo_sample.biomass_samples.min():.3f} - {zoo_sample.biomass_samples.max():.3f}")

print(f"\nP/B Ratio:")
print(f"  Base: {params.model[params.model['Group']=='Zooplankton']['PB'].values[0]:.3f} /year")
print(f"  Mean (samples): {zoo_sample.pb_samples.mean():.3f} ± {zoo_sample.pb_samples.std():.3f}")
print(f"  Range: {zoo_sample.pb_samples.min():.3f} - {zoo_sample.pb_samples.max():.3f}")

print(f"\nEcotrophic Efficiency:")
print(f"  Base: {params.model[params.model['Group']=='Zooplankton']['EE'].values[0]:.3f}")
print(f"  Mean (samples): {zoo_sample.ee_samples.mean():.3f} ± {zoo_sample.ee_samples.std():.3f}")
print(f"  Range: {zoo_sample.ee_samples.min():.3f} - {zoo_sample.ee_samples.max():.3f}")

## Cell 6: Run Ensemble

Execute 50 replicate Ecopath models with sampled parameters.

In [ ]:
import time

# Run ensemble
print(f"Running ensemble with {n_replicates} replicates...")
print(f"Groups sampled: {', '.join(groups_to_sample)}")
print()

start_time = time.time()

result = run_ensemble_rpath(
    base_params=params,
    sample_container=container,
    eco_name="Example Coastal Ecosystem",
    eco_area=1.0,
    random_seed=42,
    verbose=False
)

elapsed = time.time() - start_time

print(f"✓ Ensemble complete!")
print(f"  Total time: {elapsed:.2f} seconds")
print(f"  Total models run: {len(result.replicate_models)} (1 base + {result.n_replicates} replicates)")
print(f"  Average time per replicate: {(elapsed / result.n_replicates)*1000:.1f} ms")

## Cell 7: Extract Confidence Intervals

Calculate 95% CI for all groups and key parameters.

In [ ]:
# Get confidence intervals
ci_df = get_ensemble_confidence_intervals(result, alpha=0.05)

print("95% Confidence Intervals (Long Format):")
print("="*60)
display(ci_df.head(15))

# Filter to biomass only for easier reading
biomass_ci = ci_df[ci_df["Parameter"] == "Biomass"].copy()
biomass_ci = biomass_ci.sort_values("Mean", ascending=False)

print("\nBiomass Confidence Intervals (sorted by mean):")
print("="*60)
display(biomass_ci.round(4))

## Cell 8: Calculate System-Level Statistics

Compute ecosystem-wide uncertainty metrics.

In [ ]:
# System statistics
stats = ensemble_statistics(result)

print("System-Level Statistics:")
print("="*60)
print(f"Number of groups: {stats['n_groups']}")
print(f"Number of replicates: {stats['n_replicates']}")
print()
print("Total System Biomass:")
print(f"  Mean: {stats['total_system_biomass_mean']:.2f} t/km²")
print(f"  Std Dev: {stats['total_system_biomass_std']:.2f} t/km²")
print(f"  CV (Coefficient of Variation): {stats['total_system_biomass_cv']:.3f}")
print()
print("Mean Trophic Level:")
print(f"  Mean: {stats['mean_trophic_level_mean']:.2f}")
print(f"  Std Dev: {stats['mean_trophic_level_std']:.3f}")
print()
print("Interpretation:")
print(f"  CV < 0.1 = low uncertainty")
print(f"  CV 0.1-0.3 = moderate uncertainty")
print(f"  CV > 0.3 = high uncertainty")
print(f"  → System biomass CV of {stats['total_system_biomass_cv']:.3f} indicates moderate uncertainty")

## Cell 9: Visualization 1 - Confidence Intervals

Plot biomass confidence intervals with error bars.

In [ ]:
# Prepare data for plotting
plot_data = biomass_ci.copy()
plot_data['ErrorLow'] = plot_data['Mean'] - plot_data['LCI']
plot_data['ErrorHigh'] = plot_data['UCI'] - plot_data['Mean']

# Create plot
fig, ax = plt.subplots(figsize=(12, 6))

x_pos = np.arange(len(plot_data))
ax.errorbar(
    x_pos,
    plot_data['Mean'],
    yerr=[plot_data['ErrorLow'], plot_data['ErrorHigh']],
    fmt='o',
    capsize=5,
    markersize=10,
    linewidth=2,
    color='steelblue',
    ecolor='steelblue',
    alpha=0.7,
    label='95% Confidence Interval'
)

# Formatting
ax.set_xticks(x_pos)
ax.set_xticklabels(plot_data['Group'], rotation=45, ha='right')
ax.set_ylabel('Biomass (t/km²)', fontsize=12)
ax.set_title('Ensemble Biomass Confidence Intervals (95% CI)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Plot shows:")
print("  - Points: Mean biomass from ensemble")
print("  - Error bars: 95% confidence interval bounds")
print("  - Wider bars: More parameter uncertainty")

## Cell 10: Visualization 2 - Uncertainty Distribution

Plot percentile bands showing full distribution of outcomes.

In [ ]:
# Get percentiles for first sampled group (Zooplankton)
group_name = "Zooplankton"
percentiles = result.biomass_percentiles(
    group=group_name,
    percentiles=(2.5, 25, 50, 75, 97.5)
)

# Create visualization
fig, ax = plt.subplots(figsize=(10, 6))

perc_values = [2.5, 25, 50, 75, 97.5]
biomass_values = [percentiles[p] for p in perc_values]
labels = [f"{p}%" for p in perc_values]

# Plot bands
colors = ['red', 'orange', 'green', 'orange', 'red']
bars = ax.bar(labels, biomass_values, color=colors, alpha=0.7, edgecolor='black')

# Add value labels
for bar, val in zip(bars, biomass_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.2f}',
            ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Biomass (t/km²)', fontsize=12)
ax.set_title(f'{group_name}: Percentile Distribution', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(biomass_values) * 1.15)
ax.grid(axis='y', alpha=0.3)

# Add interpretation text
textstr = ('Interpretation:\n'
           '• Green (50%): Median - most likely value\n'
           '• Orange (25%, 75%): Interquartile range\n'
           '• Red (2.5%, 97.5%): 95% Confidence interval')
ax.text(0.98, 0.97, textstr, transform=ax.transAxes,
        fontsize=10, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Cell 11: Reproducibility Demo

Verify that setting random seed produces identical results.

In [ ]:
# Re-run ensemble with same seed
print("Re-running ensemble with same random seed (42)...")

# Create fresh container
np.random.seed(42)
container2 = SampleContainer(n_replicates=n_replicates, random_seed=42)
for group_name in groups_to_sample:
    group_row = params.model[params.model["Group"] == group_name].iloc[0]
    biomass_samples = np.random.lognormal(
        mean=np.log(group_row["Biomass"]),
        sigma=sampling_cv,
        size=n_replicates
    )
    pb_samples = np.random.lognormal(
        mean=np.log(group_row["PB"]),
        sigma=sampling_cv,
        size=n_replicates
    )
    qb_samples = np.random.lognormal(
        mean=np.log(group_row["QB"]),
        sigma=sampling_cv,
        size=n_replicates
    )
    ee_samples = np.random.beta(
        a=group_row["EE"] * (1 - group_row["EE"]) / (sampling_cv**2) - 1,
        b=(1 - group_row["EE"]) * (1 - group_row["EE"]) / (sampling_cv**2) - 1,
        size=n_replicates
    ).clip(0, 1)
    sample = EcopathGroupSample(
        group_name=group_name,
        biomass_samples=biomass_samples,
        pb_samples=pb_samples,
        qb_samples=qb_samples,
        ee_samples=ee_samples,
        biomass_se=sampling_cv,
        pb_se=sampling_cv,
    )
    container2.add_group_sample(sample)

# Run second ensemble
result2 = run_ensemble_rpath(
    base_params=params,
    sample_container=container2,
    eco_name="Example Coastal Ecosystem",
    eco_area=1.0,
    random_seed=42,
    verbose=False
)

# Compare results
ci1 = get_ensemble_confidence_intervals(result, alpha=0.05)
ci2 = get_ensemble_confidence_intervals(result2, alpha=0.05)

# Check if identical
is_identical = np.allclose(ci1['Mean'].values, ci2['Mean'].values)
print(f"\nResults identical with same seed: {is_identical}")

# Show comparison
comparison = pd.DataFrame({
    'Group': ci1['Group'],
    'Parameter': ci1['Parameter'],
    'Run1_Mean': ci1['Mean'],
    'Run2_Mean': ci2['Mean'],
    'Difference': np.abs(ci1['Mean'].values - ci2['Mean'].values)
})

print("\nComparison of two runs with same seed (first 6 rows):")
display(comparison.head(6).round(6))

print("\n✓ Reproducibility verified!")

## Cell 12: Conclusion & Next Steps

Summary and recommendations for further analysis.

In [ ]:
print("="*60)
print("ENSEMBLE ANALYSIS SUMMARY")
print("="*60)
print()
print("What we accomplished:")
print("  1. ✓ Loaded a 12-group coastal ecosystem model")
print(f"  2. ✓ Created samples for {len(container.group_samples)} key groups")
print(f"  3. ✓ Ran {result.n_replicates} ensemble replicates")
print(f"  4. ✓ Computed 95% confidence intervals for all parameters")
print(f"  5. ✓ Generated system-level uncertainty metrics")
print(f"  6. ✓ Visualized uncertainty distributions")
print()
print("Key Results:")
print(f"  - System biomass CV: {stats['total_system_biomass_cv']:.3f}")
print(f"  - Mean system biomass: {stats['total_system_biomass_mean']:.2f} t/km²")
print(f"  - Mean trophic level: {stats['mean_trophic_level_mean']:.2f}")
print()
print("Next Steps:")
print("  • Increase n_replicates to 500-1000 for publication")
print("  • Expand sampling to all consumer groups")
print("  • Conduct sensitivity analysis (tornado plots)")
print("  • Compare model predictions to field observations")
print("  • Export results for further analysis")
print()
print("Documentation:")
print("  • Usage Guide: docs/ENSEMBLE_USAGE_GUIDE.md")
print("  • Integration Tutorial: docs/ENSEMBLE_INTEGRATION_TUTORIAL.md")
print("  • API Reference: pypath.core.ensemble")
print()
print("="*60)